# 8.4 - Advanced Retrieval: Beyond Naive Cosine

**Module 8 - RAG Systems** | Netsetos GenAI Engineering

This notebook builds the retrieval engineering that turns "the right document exists" into "the right document is at rank 1." You code the four load-bearing layers from scratch - hybrid search (BM25 + dense), cross-encoder reranking, HyDE, and multi-query - then walk lighter sketches of the reranker landscape, query transforms, self-correcting RAG, and the order-aware metrics that tell you which layer actually earned its latency.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Install the libraries this notebook leans on. Uncomment the pip line on Colab or a fresh environment; skip it if your kernel already has these packages.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install google-genai numpy sentence-transformers python-dotenv -q  # noqa


**Explanation**

One-time environment prep - it installs the Gemini SDK, numpy, sentence-transformers (for the cross-encoder rerankers) and python-dotenv, and runs no code of its own.

**How the code works - step by step**
- **`google-genai`** - the Gemini client used for embeddings and the LLM query transforms.
- **`numpy`** - vector math for cosine similarity throughout.
- **`sentence-transformers`** - supplies the `CrossEncoder` rerankers in Concepts 2 and 5.
- **`python-dotenv`** - lets you load keys from a `.env` file instead of hardcoding them.

**What you'll see:** (no output - environment prep)

## Setup - load your API key

The embedding and LLM cells read `GOOGLE_API_KEY` from the environment. Set it here (or via a `.env` file) - never hardcode a key in the notebook.

> **Needs a Google AI Studio key** (get one at aistudio.google.com).

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

Environment prep, not a call - `setdefault` only fills in the variable if it is not already set, so an existing shell/`.env` value wins. Any one provider key is enough to run the embedding and generation cells.

**How the code works - step by step**
- **`os.environ.setdefault("GOOGLE_API_KEY", "")`** - registers the key name in the environment without overwriting a value you already exported.

**In one line:** make sure `GOOGLE_API_KEY` exists before the client reads it.

**What you'll see:** (no output - environment prep)

## 1 - Hybrid search: BM25 + dense embeddings

Dense embeddings blur rare exact tokens - an error code, a surname - into their neighborhood; BM25 keyword scoring nails those tokens but misses paraphrases. Run both and fuse with Reciprocal Rank Fusion, and each retriever covers the other's blind spot. This is the single biggest jump over naive dense retrieval.

> **Needs a Google AI Studio key** - the dense side makes real Gemini embedding calls.

In [ ]:
# HYBRID SEARCH - BM25 (keyword) + Dense (semantic)
# pip install google-genai numpy
import math
from collections import Counter
import numpy as np
from google import genai
from google.genai import types

client = genai.Client()  # reads GOOGLE_API_KEY from the environment

# ── BM25 (keyword search from scratch) ──
class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.docs = [d.lower().split() for d in docs]
        self.k1, self.b = k1, b
        self.avg_dl = sum(len(d) for d in self.docs) / len(self.docs)
        self.df = Counter()
        for d in self.docs:
            for w in set(d): self.df[w] += 1
        self.N = len(self.docs)

    def score(self, query):
        q_words = query.lower().split()
        scores = []
        for doc in self.docs:
            s = 0
            tf = Counter(doc)
            dl = len(doc)
            for w in q_words:
                if w not in tf: continue
                idf = math.log((self.N - self.df[w] + 0.5) / (self.df[w] + 0.5) + 1)
                tf_norm = (tf[w] * (self.k1+1)) / (tf[w] + self.k1*(1-self.b+self.b*dl/self.avg_dl))
                s += idf * tf_norm
            scores.append(s)
        return scores

# ── Dense search (cosine similarity) ──
def dense_search(query_emb, doc_embs):
    return [np.dot(query_emb,d)/(np.linalg.norm(query_emb)*np.linalg.norm(d)) for d in doc_embs]

# ── Hybrid: combine with Reciprocal Rank Fusion (RRF) ──
def hybrid_rrf(bm25_scores, dense_scores, k=60):
    """Reciprocal Rank Fusion: merge two ranked lists."""
    bm25_rank = sorted(range(len(bm25_scores)), key=lambda i:-bm25_scores[i])
    dense_rank = sorted(range(len(dense_scores)), key=lambda i:-dense_scores[i])
    rrf = {}
    for rank, idx in enumerate(bm25_rank):
        rrf[idx] = rrf.get(idx, 0) + 1/(k + rank + 1)
    for rank, idx in enumerate(dense_rank):
        rrf[idx] = rrf.get(idx, 0) + 1/(k + rank + 1)
    return sorted(rrf.items(), key=lambda x:-x[1])

# ── Demo ──
docs = [
    "Refund policy: full refund within 7 days of purchase.",
    "Error code NTS-4021: authentication token expired. Re-login required.",
    "The GenAI course covers transformers, RAG, and fine-tuning.",
    "Return and refund requests can be submitted via support@netsetos.com.",
    "NTS-4021 troubleshooting: clear browser cache, then re-authenticate.",
]
bm25 = BM25(docs)

# Real dense embeddings (one batched call) - so the dense side is meaningful,
# not random noise. task_type matters: docs vs query embed differently.
doc_embs = [np.array(e.values) for e in client.models.embed_content(
    model="gemini-embedding-001", contents=docs,
    config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")).embeddings]

def embed_query(q):
    r = client.models.embed_content(model="gemini-embedding-001", contents=q,
            config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"))
    return np.array(r.embeddings[0].values)

for query in ["how to get my money back", "NTS-4021"]:
    bm25_s = bm25.score(query)
    dense_s = dense_search(embed_query(query), doc_embs)
    hybrid = hybrid_rrf(bm25_s, dense_s)
    print(f"Query: {query!r}")
    print(f"  BM25 top:   [{np.argmax(bm25_s)}] {docs[np.argmax(bm25_s)][:52]}")
    print(f"  Hybrid top: [{hybrid[0][0]}] {docs[hybrid[0][0]][:52]}")

# Output: (BM25 nails the exact code; dense catches the paraphrase; hybrid gets both)
#   Query: 'how to get my money back'
#     BM25 top:   [3] Return and refund requests can be submitted via su
#     Hybrid top: [0] Refund policy: full refund within 7 days of purcha
#   Query: 'NTS-4021'
#     BM25 top:   [1] Error code NTS-4021: authentication token expired.
#     Hybrid top: [1] Error code NTS-4021: authentication token expired.

**Explanation**

A from-scratch implementation, not a library call: it builds BM25 keyword scoring and cosine dense search by hand, then fuses their rankings with RRF. Reading it is the fastest way to see why fusion beats either retriever alone - and why RRF looks at rank position, never the incompatible raw scores.

**How the code works - step by step**
- **`BM25.__init__`** - lowercases and tokenizes every doc, precomputes average document length, per-word document frequency, and corpus size.
- **`BM25.score`** - for a query, sums the classic BM25 term score per doc (idf times a length-normalized, saturating term frequency).
- **`dense_search`** - cosine similarity between the query embedding and each document embedding.
- **`hybrid_rrf`** - ranks docs by each method, awards each doc `1/(60+rank)` from BOTH lists, sums, and sorts - so a doc ranked high by EITHER retriever floats up.
- **demo + real embeddings** - embeds the 5 docs once as `RETRIEVAL_DOCUMENT` and each query as `RETRIEVAL_QUERY` (task_type matters), so the dense side is meaningful, not noise.
- **query loop** - prints BM25's top doc versus hybrid's top doc for a paraphrase and an exact code.

**In one line:** keyword score + cosine score -> fuse by rank -> best of both.

**What you'll see:** For "how to get my money back" BM25 tops the literal "Return and refund..." doc [3] while hybrid promotes the refund-policy doc [0]; for the exact code "NTS-4021" both BM25 and hybrid land on doc [1]. Exact match and paraphrase are both covered.

## 2 - Re-ranking with a cross-encoder

The retriever you built is a bi-encoder: it embeds query and document separately, so it is fast but never reads them together. A cross-encoder reads the pair jointly and is far more accurate, but must run once per candidate - so you retrieve a wide shortlist cheaply, then rerank just those.

> **Downloads a small model on first run** (BAAI/bge-reranker-v2-m3 via sentence-transformers).

In [ ]:
# RE-RANKING with a real cross-encoder (retrieve wide, rerank narrow)
# pip install sentence-transformers
from sentence_transformers import CrossEncoder

# A cross-encoder reads query AND document TOGETHER and outputs one relevance score.
# Far more accurate than bi-encoder cosine, but must run once per candidate -
# so you rerank a SHORTLIST (top-k from retrieval), never the whole corpus.
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")  # open, multilingual, self-hostable

def rerank(query, candidates, top_k=3):
    pairs = [(query, doc) for doc in candidates]
    scores = reranker.predict(pairs)  # one score per (query, doc) pair
    ranked = sorted(zip(candidates, scores), key=lambda x: -x[1])
    return ranked[:top_k]

# Demo: 5 candidates from a bi-encoder shortlist, reranked to top 3
query = "What are the prerequisites for the GenAI course?"
candidates = [
    "The GenAI course costs 14999 rupees with lifetime access.",
    "Prerequisites: basic Python programming and high school math.",
    "Live classes are held daily at 7 PM IST on YouTube.",
    "Students should know Python basics including loops, functions, and lists.",
    "Our refund policy allows full refunds within 7 days.",
]

print("After reranking (top 3):")
for doc, score in rerank(query, candidates):
    print(f"  {score:+.2f}  {doc[:56]}")

# Output: (the two prerequisites docs rise; pricing/refund/schedule sink)
#   After reranking (top 3):
#     +7.14  Prerequisites: basic Python programming and high school ma
#     +5.02  Students should know Python basics including loops, functi
#     -3.88  The GenAI course costs 14999 rupees with lifetime access.

**Explanation**

The two-stage retrieve-then-rerank move in a few lines: a real cross-encoder scores each (query, doc) pair jointly and reorders a 5-doc shortlist. Unlike the cosine bi-encoder, it can tell a subtly-wrong doc from the true answer because attention flows across both texts.

**How the code works - step by step**
- **`CrossEncoder("BAAI/bge-reranker-v2-m3")`** - loads an open, multilingual, self-hostable reranker.
- **`rerank`** - builds `(query, doc)` pairs, predicts one relevance score each, sorts descending, keeps `top_k`.
- **demo** - 5 candidates standing in for a bi-encoder shortlist (two about prerequisites, three off-topic).
- **print loop** - shows the top 3 docs with their signed scores.

**In one line:** score each pair jointly -> sort -> keep the best few.

**What you'll see:** The two prerequisites docs rise to the top (+7.14 and +5.02) while the pricing doc sinks to -3.88. The first run downloads the bge-reranker model.

## 3 - HyDE: hypothetical document embeddings

A short question and a real answer document are phrased differently, so they sit apart in embedding space. HyDE asks the LLM to first hallucinate a plausible answer, then embeds THAT as the search key - because answers cluster with answers, the fake answer lands near the real documents. The user never sees the hallucination.

> **Needs a Google AI Studio key** - one generation call plus embedding calls.

In [ ]:
# HyDE - Hypothetical Document Embeddings
# pip install google-genai numpy
from google import genai
from google.genai import types
import numpy as np

client = genai.Client()  # reads GOOGLE_API_KEY from the environment

def embed(text, task="RETRIEVAL_QUERY"):
    r = client.models.embed_content(model="gemini-embedding-001", contents=text,
            config=types.EmbedContentConfig(task_type=task))
    return np.array(r.embeddings[0].values)

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

class HyDERetriever:
    """Generate a hypothetical answer, embed THAT, and search with it."""
    def __init__(self, chunks, embeddings):
        self.chunks, self.embs = chunks, embeddings

    def retrieve_naive(self, query, k=2):
        qe = embed(query, "RETRIEVAL_QUERY")
        scores = sorted(((i, cosine(qe, e)) for i, e in enumerate(self.embs)), key=lambda x: -x[1])
        return [(self.chunks[i], sc) for i, sc in scores[:k]]

    def retrieve_hyde(self, query, k=2):
        # 1. Ask the LLM for a plausible answer (may be factually WRONG - that is fine)
        hypo = client.models.generate_content(model="gemini-2.5-flash",
            contents=f"Write a short company-FAQ paragraph answering: {query}").text.strip()
        # 2. Embed the hypothetical ANSWER, not the question - answers cluster with answers
        hyde_emb = embed(hypo, "RETRIEVAL_QUERY")
        scores = sorted(((i, cosine(hyde_emb, e)) for i, e in enumerate(self.embs)), key=lambda x: -x[1])
        return [(self.chunks[i], sc) for i, sc in scores[:k]], hypo

chunks = [
    "Refund policy: full refund within 7 days. 50 percent from 8-30 days.",
    "GenAI course: 14999 rupees, lifetime access, Discord, GCP credits.",
    "Live classes daily 7 PM IST. Recordings in 2 hours. Python + GCP.",
    "Prerequisites: basic Python and high school math. No ML experience needed.",
]
embs = [embed(c, "RETRIEVAL_DOCUMENT") for c in chunks]
hyde = HyDERetriever(chunks, embs)

query = "can I get my money back?"
naive = hyde.retrieve_naive(query)
hyde_res, hypo = hyde.retrieve_hyde(query)
print(f"Naive top: [{naive[0][1]:.3f}] {naive[0][0][:52]}")
print(f"HyDE  top: [{hyde_res[0][1]:.3f}] {hyde_res[0][0][:52]}")

# Output: (HyDE ranks the refund doc higher - its fake answer LOOKS like a policy)
#   Naive top: [0.612] Refund policy: full refund within 7 days. 50 perce
#   HyDE  top: [0.771] Refund policy: full refund within 7 days. 50 perce

**Explanation**

A side-by-side comparison, not just a demo: it runs naive query-embedding retrieval and HyDE retrieval on the same query so you can watch the cosine score move. The generated answer may be factually wrong - it is used only as a retrieval key.

**How the code works - step by step**
- **`embed` / `cosine`** - helpers to embed text with a chosen task_type and score two vectors.
- **`HyDERetriever.retrieve_naive`** - embeds the raw query, cosine against all docs, returns top-k.
- **`HyDERetriever.retrieve_hyde`** - asks gemini-2.5-flash for a plausible FAQ paragraph, embeds THAT paragraph, then cosine-searches with it (and returns the hypothetical).
- **corpus** - 4 chunks embedded once as `RETRIEVAL_DOCUMENT`.
- **run + print** - compares the top hit and score from naive vs HyDE on "can I get my money back?".

**In one line:** hallucinate an answer -> embed the fake answer -> search with it.

**What you'll see:** Both retrievers pick the refund doc, but HyDE raises its cosine from about 0.612 to about 0.771 - the fake answer looks like a policy, so it matches the real policy better.

## 4 - Multi-query retrieval

One phrasing of a question only matches documents phrased similarly. Multi-query asks the LLM for several rephrasings, retrieves for each, and merges with RRF - so a document that only matches one angle still surfaces. It widens the net (recall); it does not sharpen any single match.

> **Needs a Google AI Studio key** - one generation call plus embedding calls.

In [ ]:
# MULTI-QUERY RETRIEVAL - several phrasings, merged with RRF
# pip install google-genai numpy
from google import genai
from google.genai import types
import numpy as np

client = genai.Client()

def embed(t, task="RETRIEVAL_QUERY"):
    r = client.models.embed_content(model="gemini-embedding-001", contents=t,
            config=types.EmbedContentConfig(task_type=task))
    return np.array(r.embeddings[0].values)

def cosine(a, b): return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def rrf(rank_lists, k=60):
    """Reciprocal Rank Fusion - reward docs ranked high by ANY phrasing."""
    scores = {}
    for ranks in rank_lists:
        for pos, doc_id in enumerate(ranks):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + pos + 1)
    return sorted(scores, key=scores.get, reverse=True)

class MultiQueryRetriever:
    def __init__(self, chunks, embeddings):
        self.chunks, self.embs = chunks, embeddings

    def expand_query(self, query, n=3):
        resp = client.models.generate_content(model="gemini-2.5-flash",
            contents=f"Give {n} alternative phrasings of this question, one per line, questions only.\n\n{query}")
        alts = [l.strip().lstrip("0123456789.-) ") for l in resp.text.strip().split("\n") if l.strip()]
        return [query] + alts[:n]

    def rank_for(self, q):
        qe = embed(q, "RETRIEVAL_QUERY")
        return [i for i, _ in sorted(enumerate(self.embs), key=lambda ie: -cosine(qe, ie[1]))]

    def retrieve(self, query, k=2):
        queries = self.expand_query(query)
        fused = rrf([self.rank_for(q) for q in queries])  # RRF across every phrasing
        return [self.chunks[i] for i in fused[:k]], queries

chunks = [
    "Refund policy: full refund within 7 days. 50 percent from 8-30 days.",
    "GenAI course: 14999 rupees, lifetime access, Discord, GCP credits.",
    "Live classes daily 7 PM IST. Recordings in 2 hours.",
    "Prerequisites: basic Python and high school math.",
]
embs = [embed(c, "RETRIEVAL_DOCUMENT") for c in chunks]
mq = MultiQueryRetriever(chunks, embs)

results, expanded = mq.retrieve("what do I need to know before starting?")
print("Expanded to:", len(expanded), "phrasings")
for c in results: print(f"  {c[:56]}")

# Output:
#   Expanded to: 4 phrasings
#   Prerequisites: basic Python and high school math.
#   Live classes daily 7 PM IST. Recordings in 2 hours.

**Explanation**

A recall-widening pipeline built around RRF again: generate alternative phrasings, rank the corpus independently for each, and fuse. The fusion rewards docs ranked high by ANY phrasing, using rank position rather than raw scores.

**How the code works - step by step**
- **`embed` / `cosine` / `rrf`** - helpers; `rrf` sums `1/(60+pos)` for each doc across every rank list.
- **`MultiQueryRetriever.expand_query`** - asks gemini-2.5-flash for 3 alternative phrasings, cleans the list, and prepends the original query.
- **`rank_for`** - returns the full doc ranking for a single phrasing by cosine.
- **`retrieve`** - ranks the corpus for every phrasing, RRF-fuses the lists, returns top-k.
- **run + print** - expands "what do I need to know before starting?" and prints the phrasing count and merged results.

**In one line:** rephrase x4 -> rank each -> RRF-fuse.

**What you'll see:** The original query expands to 4 phrasings, and the merged top-2 surface the prerequisites doc and the live-classes doc - documents that a single phrasing might have missed.

## 5 - The reranker landscape in one call shape

Concept 2 built one reranker; production teams choose among a dozen, trading accuracy, latency, cost, language coverage, and self-hosting. The point here: the call shape is identical across tiers - you swap the model id to move the accuracy/latency dial.

> **Downloads a small model on first run** (bge-reranker-v2-m3).

In [ ]:
# Same call shape across the tiers - swap the model to move the accuracy/latency dial
# pip install sentence-transformers
from sentence_transformers import CrossEncoder

TIERS = {
    "accurate": "BAAI/bge-reranker-v2-m3",     # open, multilingual, self-host
    "fast":     "cross-encoder/ms-marco-MiniLM-L-6-v2",  # lighter, quicker
}

def rerank(model_id, query, docs, top_k=3):
    ce = CrossEncoder(model_id)
    scores = ce.predict([(query, d) for d in docs])
    return sorted(zip(docs, scores), key=lambda x: -x[1])[:top_k]

# ColBERT-style "late interaction" is the middle tier: precompute token-level
# vectors per doc, then do the fine per-token MaxSim comparison cheaply at query time -
# near cross-encoder quality, closer to bi-encoder speed. Cohere/Jina are hosted APIs.
print(rerank(TIERS["accurate"], "prerequisites?", ["Python + math", "costs 14999"])[0][0])

# Output:
#   Python + math

**Explanation**

A parameterized version of the Concept 2 reranker: one `rerank` function, a `TIERS` map of model ids, and a comment placing ColBERT-style late interaction and hosted Cohere/Jina APIs on the same dial. It makes the tier choice concrete rather than a marketing table.

**How the code works - step by step**
- **`TIERS`** - maps "accurate" (bge-reranker-v2-m3) and "fast" (ms-marco-MiniLM-L-6-v2) to their model ids.
- **`rerank`** - loads the chosen `CrossEncoder`, scores `(query, doc)` pairs, sorts, keeps `top_k`.
- **comment block** - explains ColBERT's late-interaction middle tier (precomputed token vectors, cheap MaxSim at query time) and that Cohere/Jina are hosted APIs.
- **print** - reranks a 2-doc demo with the accurate tier.

**In one line:** one function, swap the model id to trade accuracy for speed.

**What you'll see:** Prints "Python + math" - the relevant doc beats "costs 14999". The first run downloads the bge-reranker model.

## 6 - Query transformation templates

HyDE and multi-query were two query transforms; this completes the toolbox. Step-back abstracts a narrow question to a broader one, decomposition splits a multi-hop question into sub-questions, and routing decides how to handle a query at all. The skill is diagnosing which transform a query shape needs.

In [ ]:
# The transforms are just prompts - the skill is choosing which one
STEP_BACK = "Rewrite this narrow question as the broader question whose answer contains it.\n{q}"
DECOMPOSE = "Break this into the minimal set of standalone sub-questions, one per line.\n{q}"
ROUTE     = "Answer with one word - RETRIEVE, WEB, or DIRECT - for how to handle:\n{q}"

# Examples of what each returns:
#   step-back:  "2019 amendment to clause 4b?"  ->  "How is clause 4 structured?"
#   decompose:  "compare refund vs cancellation" ->  ["refund policy?", "cancellation policy?"]
#   route:      "what is 2 + 2?"                 ->  DIRECT   (no retrieval needed)

def transform(client, template, q):
    return client.models.generate_content(model="gemini-2.5-flash",
        contents=template.format(q=q)).text.strip()

# Output: (transform("...", STEP_BACK, "2019 amendment to clause 4b?"))
#   How is clause 4 structured, and what amendments has it had?

**Explanation**

A reference cell - the three transforms are literally just prompt templates, and the point is that choosing among them is the hard part, not the code. `transform` is a thin wrapper that fills a template and calls the model; nothing executes at import.

**How the code works - step by step**
- **`STEP_BACK` / `DECOMPOSE` / `ROUTE`** - the three prompt templates (broaden, split, classify-how-to-handle).
- **comment block** - shows what each template returns on a worked example.
- **`transform`** - formats a chosen template with the query and calls gemini-2.5-flash.

**In one line:** three prompts plus one caller; routing decides which (or none) fits.

**What you'll see:** No output - the cell only defines the templates and the `transform` helper (the "# Output:" comment shows an example step-back rewrite). Calling `transform` would need a Google AI Studio key.

## 7 - CRAG: grade-then-branch

Everything so far runs retrieval once. Self-correcting architectures let the system check its own retrieval and act: retrieve again, correct with web search, or use as-is. CRAG wraps any frozen LLM with a lightweight grader and three fallback paths.

In [ ]:
# CRAG in one function: grade the retrieved docs, then take one of three paths
def crag_step(grade, docs, web_search):
    if grade == "correct":      return docs                          # use as-is
    if grade == "ambiguous":    return docs + web_search()            # augment
    return web_search()                                            # "incorrect": discard, fall back

# A lightweight grader (small model or classifier) scores retrieval quality;
# Self-RAG instead BAKES this decision into the model via reflection tokens
# (needs fine-tuning); Adaptive-RAG routes by predicted question difficulty.
print(crag_step("incorrect", [], lambda: ["fresh web result"]))

# Output:
#   ['fresh web result']

**Explanation**

A control-flow sketch, not a model call - CRAG's whole decision compressed into one branching function. It shows the three-action routing (correct / ambiguous / incorrect) that a separate grader feeds; Self-RAG bakes the same decision into a fine-tuned model instead.

**How the code works - step by step**
- **`crag_step`** - branches on the grade: "correct" returns the docs as-is; "ambiguous" augments them with a web search; anything else ("incorrect") discards the docs and falls back to web search only.
- **comment** - notes that Self-RAG bakes this into the model via reflection tokens and Adaptive-RAG routes by predicted question difficulty.
- **print** - runs the "incorrect" path with empty docs and a stubbed web-search function.

**In one line:** grade -> keep / augment / replace.

**What you'll see:** Prints ['fresh web result'] - the "incorrect" grade discards the (empty) retrieved docs and falls back entirely to the web-search result.

## 8 - Order-aware metrics by hand: MRR & NDCG

Every layer in this lesson claims an uplift, but on your corpus some help and some just add latency - evaluation is how you tell. The key distinction: order-aware metrics (MRR, NDCG) reward putting the right document HIGH, while recall@k only asks whether it is in the top k at all.

In [ ]:
# NDCG and MRR worked by hand on one query - see why order matters
import math

# The one relevant doc is at rank 3 (0-indexed position 2) in our top-5
relevance = [0, 0, 1, 0, 0]   # 1 = the right answer

# MRR: reciprocal of the FIRST relevant rank -> 1/3
first = next(i for i, r in enumerate(relevance) if r) + 1
mrr = 1 / first

# NDCG@5: DCG = sum(rel / log2(rank+1)); normalize by the ideal (answer at rank 1)
dcg   = sum(r / math.log2(i + 2) for i, r in enumerate(relevance))
ideal = 1 / math.log2(1 + 1)   # the one relevant doc, placed at rank 1
print(f"MRR = {mrr:.2f}   NDCG@5 = {dcg/ideal:.2f}")

# recall@5 would be 1.0 here (the answer IS in the top 5) - it cannot tell rank 3
# from rank 1. MRR and NDCG can, which is why they are the order-aware metrics.

# Output:
#   MRR = 0.33   NDCG@5 = 0.50

**Explanation**

A measurement harness, not a model call. It works MRR and NDCG@5 by hand on one query where the answer sits at rank 3, then points out that recall@5 would score a perfect 1.0 there - making concrete why order-aware metrics exist.

**How the code works - step by step**
- **`relevance`** - a 0/1 list marking the one right doc at position 2 (rank 3) of the top-5.
- **MRR** - reciprocal of the first relevant rank, so 1/3.
- **NDCG@5** - DCG = sum of `rel / log2(rank+1)`, normalized by the ideal DCG (the answer placed at rank 1).
- **comment** - notes recall@5 would be 1.0 and cannot distinguish rank 3 from rank 1.

**In one line:** a rank-3 answer costs you on MRR/NDCG but not on recall.

**What you'll see:** Prints "MRR = 0.33   NDCG@5 = 0.50" - both penalize the rank-3 placement that recall@5 would have scored a perfect 1.0.

## 9 - Semantic caching

A four-layer retrieval stack can blow a latency budget and a bill. A large share of production traffic is near-identical queries - semantic caching answers those for free by matching embeddings, skipping retrieval and generation entirely.

In [ ]:
# Semantic cache: answer a near-identical query for free by matching its embedding
import numpy as np

def cache_lookup(q_emb, cache, threshold=0.95):
    """cache: list of (embedding, answer). Return a hit if close enough."""
    for emb, answer in cache:
        sim = float(np.dot(q_emb, emb) / (np.linalg.norm(q_emb) * np.linalg.norm(emb)))
        if sim >= threshold:
            return answer   # exact-enough match - skip retrieval + generation entirely
    return None

# "what is the refund window?" and "how many days for a refund?" embed close,
# so the second query is served from cache - a large share of production traffic repeats.

q_emb = np.array([0.1, 0.2, 0.3])   # any query vector; here it matches the cached one
print(cache_lookup(q_emb, [(q_emb, "Refunds within 7 days.")]))

# Output:
#   Refunds within 7 days.

**Explanation**

A tiny cache lookup in pure numpy, no model call. It compares a new query's embedding against cached ones and returns the stored answer when the similarity clears a threshold - the cheapest possible retrieval layer.

**How the code works - step by step**
- **`cache_lookup`** - loops the cached `(embedding, answer)` pairs, computes cosine against the query embedding, and returns the answer as soon as similarity is at or above the 0.95 threshold (else `None`).
- **comment** - notes that near-identical phrasings embed close, so repeated traffic is served without a model call.
- **demo** - passes a query vector equal to the single cached entry, guaranteeing a hit.

**In one line:** close-enough embedding -> serve the cached answer.

**What you'll see:** Prints "Refunds within 7 days." - the query vector matches the one cached entry above the threshold, so the cached answer is returned instead of re-running the pipeline.

The through-line: retrieval sets the ceiling, everything else reorders under it. Hybrid search is the non-negotiable default (dense catches meaning, BM25 catches exact tokens, RRF fuses their blind spots); reranking and query transforms are the precision and recall moves you add on top, and the metrics cell is what keeps each addition honest on your own corpus. Next, Lessons 8.5-8.6 collapse this whole stack behind LangChain/LlamaIndex interfaces, 8.10 grows the CRAG/Self-RAG loops into full agents, and 8.11 turns the NDCG/MRR/RAGAS metrics into a CI evaluation harness.